# 05 - Run (production)

The routine path. Set a date range, run top to bottom, get one workbook covering
every fixture across all five leagues.

This notebook contains no modelling logic and never re-derives anything the
research notebooks already solved -- it loads frozen artifacts and scores. It does
not touch Cleaning, Features, Tuning or Evaluation.

In [1]:
# The package is installed editable (`pip install -e .`), so this works from any
# working directory -- no `os.getcwd()` gymnastics.
import fpp
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("fpp", fpp.__version__, "| targets:", fpp.TARGETS)

fpp 0.1.0 | targets: ('goals', 'shots', 'sot', 'corners')


## 1. Date range

In [2]:
DATE_FROM = "12-09-2026"     # dd-mm-yyyy
DATE_TO   = "12-09-2026"
REFRESH   = True            # pull fresh current-season data first

## 2. Refresh

Same ingestion functions as the Data Pull notebook -- not a copy of them.

In [3]:
if REFRESH:
    print(fpp.ingest.refresh("current", date_from=DATE_FROM, date_to=DATE_TO))
else:
    print("skipped - using data already on disk")

[understat_current]


[09/12/26 13:11:20] INFO     No custom team name replacements found. You can configure these in       ]8;id=12261136;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=12261137;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_config.py#91\91]8;;\
                             /Users/patrickknott/soccerdata/config/teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=12261143;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=12261144;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_config.py#189\189]8;;\
                             /Users/patrickknott/soccerdata/config/league_dict.json.                               

[results_current]
  5 date(s) not finished yet: Prem 20260912, Liga 20260912, Bund 20260912, Serie 20260912, Ligue 20260912
  fetched 0 scoreboard file(s) from ESPN
[match_stats]
  Dropped 1,445 event(s) with all-zero stats (ESPN empty-block sentinel, not a 0-0 result)
  Dropped 1,050 event(s) with every shot on target and no corners (malformed ESPN stat block, not a played match)
  Parsed 30,825 events with full shots/SOT/corners from the cache
  90 second-tier-only club(s) unmapped -- expected, Understat does not cover those divisions; their matches are still kept for the mapped opponent
  Saved 29,130 rows -> /Users/patrickknott/Developer/proposition-portfolio/Inputs/Stats/espn_match_stats.csv
[reconcile]
  Understat: 6 season(s) to refetch


[09/12/26 13:11:34] INFO     Saving cached data to                                                   ]8;id=12261151;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=12261152;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Developer/proposition-portfolio/Inputs/soccerdata_c               
                             ache                                                                                  

...

[09/12/26 13:11:36] INFO     Saving cached data to                                                   ]8;id=12261171;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=12261172;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Developer/proposition-portfolio/Inputs/soccerdata_c               
                             ache                                                                                  

                    INFO     Saving cached data to                                                   ]8;id=12261177;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=12261178;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Developer/proposition-portfolio/Inputs/soccerdata_c               
                             ache                                                                                  

[09/12/26 13:11:37] INFO     Saving cached data to                                                   ]8;id=12261183;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=12261184;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Developer/proposition-portfolio/Inputs/soccerdata_c               
                             ache                                                                                  

[09/12/26 13:11:38] INFO     Saving cached data to                                                   ]8;id=12261189;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=12261190;file:///Users/patrickknott/Developer/proposition-portfolio/Advanced_Football_Project/lib/python3.14/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/patrickknott/Developer/proposition-portfolio/Inputs/soccerdata_c               
                             ache                                                                                  

  ESPN: fetching 4 missing date file(s) across 4 league-season(s)
  Bund 2627: 1 dates missing, 1 new files
  Liga 2627: 1 dates missing, 1 new files
  Ligue 2627: 1 dates missing, 1 new files
  Serie 2627: 1 dates missing, 1 new files
  Dropped 1,445 event(s) with all-zero stats (ESPN empty-block sentinel, not a 0-0 result)
  Dropped 1,050 event(s) with every shot on target and no corners (malformed ESPN stat block, not a played match)
  Parsed 30,825 events with full shots/SOT/corners from the cache
  90 second-tier-only club(s) unmapped -- expected, Understat does not cover those divisions; their matches are still kept for the mapped opponent
  Saved 29,130 rows -> /Users/patrickknott/Developer/proposition-portfolio/Inputs/Stats/espn_match_stats.csv
  ESPN stats joined to 20,951/21,739 matches (96.4%)
  reconciled in 15s
Reconcile: 65 league-season(s) checked, 6 Understat refetch(es), 4 scoreboard file(s) fetched
  18 still short:
    Prem 2016/2017: ESPN 4.7% (not in ESPN's data)
 

## 3. Load

In [4]:
tm  = fpp.clean.build_clean_table() if REFRESH else fpp.clean.load_clean_table()
ctx = fpp.RunContext.load()

print(f"history : {len(tm):,} team-rows to {tm['date'].max().date()}")
print(f"artifacts: {ctx.version}")
for t in fpp.TARGETS:
    print(f"  {t:8} L={ctx.windows[t]['L']:>2} alpha={ctx.windows[t]['alpha']:.2f} "
          f"{len(ctx.features[t])} features, {ctx.n_estimators[t]} trees")

  ESPN stats joined to 20,951/21,739 matches (96.4%)
  7 league-season(s) below 95% ESPN coverage:
league_key    season  matches  with_stats  pct
      Bund 2021/2022      306         273 89.2
      Liga 2022/2023      380         357 93.9
      Prem 2016/2017      380          18  4.7
      Prem 2021/2022      380         353 92.9
      Prem 2022/2023      380         297 78.2
     Serie 2022/2023      380         342 90.0
     Serie 2023/2024      380         344 90.5
  Saved 43,478 rows -> /Users/patrickknott/.cache/football_prediction/clean/team_matches_3de12cfa0c77.parquet
history : 43,478 team-rows to 2026-09-11
artifacts: v2026-08-25
  goals    L=41 alpha=0.02 5 features, 417 trees
  shots    L=50 alpha=0.10 12 features, 170 trees
  sot      L=60 alpha=0.05 4 features, 101 trees
  corners  L=35 alpha=0.05 4 features, 102 trees


## 4. Retrain on the full history

Including the seasons held out during research. That period existed to get an
honest read on the model, not to be preserved forever -- once a model has been
evaluated and accepted there is no reason to keep starving it of recent matches.

In [5]:
models = fpp.predict.train_production(tm, ctx)

  goals    trained on 39,826 rows, 5 features, 417 trees
  shots    trained on 38,258 rows, 12 features, 170 trees
  sot      trained on 38,258 rows, 4 features, 101 trees
  corners  trained on 38,258 rows, 4 features, 102 trees


## 5. Score

Upcoming fixtures are appended to the team-match table as rows with unknown
outcomes and pushed through the *same* buffer pass as history. Record-before-update
on such a row is by construction the correct as-of-now prior -- which is why there
is no separate "latest priors" lookup to drift out of sync.

**Read the coverage check.** A fixture team with no history is not an error and
does not stop the run -- it is scored from the league average alone, which looks
exactly like a real prediction and is not one. A team whose history is years old
is the quieter version of the same problem. Neither was visible before.

In [6]:
fixtures = fpp.predict.load_upcoming_fixtures(
    pd.to_datetime(DATE_FROM, format="%d-%m-%Y"),
    pd.to_datetime(DATE_TO, format="%d-%m-%Y"),
)
print(f"{len(fixtures)} fixtures in range")

coverage = fpp.predict.check_fixture_coverage(fixtures, tm)

preds = fpp.predict.score_fixtures(tm, fixtures, models)
cols = ["date", "league", "home_team", "away_team",
        "goals_home", "goals_away", "shots_home", "shots_away", "corners_home", "corners_away"]
display(preds[cols].head(12))

25 fixtures in range
  all 50 fixture teams have history within 2 season(s)


,date,league,home_team,away_team,goals_home,goals_away,shots_home,shots_away,corners_home,corners_away
0,2026-09-12,Bundesliga,Augsburg,Bayer Leverkusen,1.670227,1.723202,15.342406,14.454850,5.168942,5.080503
1,2026-09-12,Bundesliga,Borussia Dortmund,Paderborn,2.697721,0.708855,18.060987,8.835456,6.638475,3.085337
2,2026-09-12,Bundesliga,FC Cologne,Werder Bremen,1.815857,1.194787,16.059566,10.785251,6.300621,4.090548
3,2026-09-12,Bundesliga,Freiburg,Borussia M.Gladbach,1.811369,1.191558,15.877456,11.683873,5.465029,4.003983
4,2026-09-12,Bundesliga,Hoffenheim,VfB Stuttgart,1.869716,1.774434,15.839042,12.193990,6.276272,4.739335
5,2026-09-12,Bundesliga,Mainz 05,Eintracht Frankfurt,1.813244,1.448993,17.073069,10.834668,5.752502,4.077818
6,2026-09-12,La Liga,Athletic Club,Elche,1.777104,0.864120,15.496094,9.239459,6.687833,3.672979
7,2026-09-12,La Liga,Osasuna,Espanyol,1.636332,0.980194,16.398474,9.870667,5.739641,3.638118
8,2026-09-12,La Liga,Racing Santander,Alaves,1.623300,1.076047,12.617376,13.094423,5.668694,4.569641
9,2026-09-12,La Liga,Real Madrid,Rayo Vallecano,2.586058,0.963150,19.479753,10.260548,7.076585,4.142270


## 6. Write the workbook

One file, every league. Colour coding compares each probability to **that league's
own** realised rate -- the model is pooled, the comparison baseline is not.

The same numbers also go out as JSON for the Edge Book app. It is written *here*
rather than in `06_Split` because this is the only notebook that has them:
`06_Split` reads the filled odds form and nothing else by design, so the
scoreline matrix, the 1X2/BTTS markets and the projected rates are not
recoverable there without retraining or a string join back to this workbook.

In [7]:
path = fpp.report.write_workbook(preds, tm, dispersion=ctx.dispersion)
print("\n->", path)

# The Match Board's whole input. Same sheet codes, same ladders, same league
# baselines as the workbook above -- one derivation, two renderings.
pred_json = fpp.report.write_predictions_json(preds, tm, dispersion=ctx.dispersion)
print("->", pred_json)

Wrote 25 fixture sheets -> /Users/patrickknott/Developer/proposition-portfolio/Outputs/predictions_2026-09-12.xlsx

-> /Users/patrickknott/Developer/proposition-portfolio/Outputs/predictions_2026-09-12.xlsx
Wrote 25 fixtures -> /Users/patrickknott/Developer/proposition-portfolio/Outputs/Data/predictions_2026-09-12.json
-> /Users/patrickknott/Developer/proposition-portfolio/Outputs/Data/predictions_2026-09-12.json


## 7. Odds capture form

A second workbook, same sheet names as the first, listing **every** proposition
each fixture is priced at -- the whole ladder, not just the half the model likes
-- with one blank column per book in `staking.BOOKS`.

`MIN_MODEL_P` is zero. The filter existed when the form was typed in by hand; the
`fill-odds` skill now populates it from Oddschecker, so breadth is cheap, and the
lines it used to cut are exactly where a book's margin is widest.

Fill it in, then run `06_Split`. The model's own probability travels with each
row, so that one file is all `06_Split` needs -- nothing has to be matched back
to this workbook afterwards.

In [8]:
props = fpp.staking.propositions(preds, ctx.dispersion)
qual = fpp.staking.qualifying(props)

n_books = len(fpp.staking.BOOK_COLUMNS)
print(f"{len(props)} propositions -> {len(qual)} at P >= {fpp.staking.MIN_MODEL_P:.2f} "
      f"({len(qual) / len(props):.1%}, {len(qual) / len(preds):.1f} per match, "
      f"{len(qual) * n_books:,} cells across {n_books} books)")

form = fpp.report.write_odds_form(qual, source=path.name)
print("\n->", form)

1850 propositions -> 1850 at P >= 0.00 (100.0%, 74.0 per match, 11,100 cells across 6 books)
Wrote 1850 propositions across 25 fixtures -> /Users/patrickknott/Developer/proposition-portfolio/Outputs/odds_input_2026-09-12.xlsx

-> /Users/patrickknott/Developer/proposition-portfolio/Outputs/odds_input_2026-09-12.xlsx
